### RAG Pipeline - data ingestion to vector

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\USER\Downloads\RAG_CRASH_COURSE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f" ✅ Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f" ❌ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all pdfs in the data dictionary
all_pdf_documents = process_all_pdfs("../data")


Found 3 PDF files to process

Processing: 23JCEIKVL65X2RGSARNY2VOUNXBTJ5AS.pdf
 ✅ Loaded 5 pages

Processing: file-example_PDF_500_kB.pdf
 ✅ Loaded 5 pages

Processing: file-sample_150kB.pdf
 ✅ Loaded 4 pages

Total documents loaded: 14


In [3]:
### Text splitting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents)
chunks

Split 14 documents into 58 chunks

Example chunk:
Content: Direct observations of N 2O5 reactivity on ambient aerosol particles
Timothy H. Bertram, 1,2 Joel A. Thornton, 1 Theran P . Riedel,3 Ann M. Middlebrook, 4
Roya Bahreini, 4,5 Timothy S. Bates, 6 Patric...
Metadata: {'producer': 'Acrobat Distiller 8.0.0 (Windows)', 'creator': '3B2 Total Publishing 6.06b/W', 'creationdate': '2009-09-29T14:05:51+08:00', 'moddate': '2009-09-30T12:20:05+08:00', 'title': 'gl040248 1..5', 'source': '..\\data\\pdf\\23JCEIKVL65X2RGSARNY2VOUNXBTJ5AS.pdf', 'total_pages': 5, 'page': 0, 'page_label': '0', 'source_file': '23JCEIKVL65X2RGSARNY2VOUNXBTJ5AS.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 8.0.0 (Windows)', 'creator': '3B2 Total Publishing 6.06b/W', 'creationdate': '2009-09-29T14:05:51+08:00', 'moddate': '2009-09-30T12:20:05+08:00', 'title': 'gl040248 1..5', 'source': '..\\data\\pdf\\23JCEIKVL65X2RGSARNY2VOUNXBTJ5AS.pdf', 'total_pages': 5, 'page': 0, 'page_label': '0', 'source_file': '23JCEIKVL65X2RGSARNY2VOUNXBTJ5AS.pdf', 'file_type': 'pdf'}, page_content='Direct observations of N 2O5 reactivity on ambient aerosol particles\nTimothy H. Bertram, 1,2 Joel A. Thornton, 1 Theran P . Riedel,3 Ann M. Middlebrook, 4\nRoya Bahreini, 4,5 Timothy S. Bates, 6 Patricia K. Quinn, 6 and Derek J. Coffman 6\nReceived 28 July 2009; accepted 10 September 2009; published 8 October 2009.\n[1]N 2O5 reactivity has been measured directly for the first\ntime on ambient aerosol particles using an entrained aerosol\nflow reactor coupled to a custom-built chemical ionization\nmass spectrometer at two urban locations during summer.\nThe observed N

### Embedding & VectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name = HuggingFace model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise

    def generate_embeddings(self, texts: List[str])->np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    # def get_embedding_dimension(self)->int:
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_sentence_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager


Loading Embedding Model: all-MiniLM-L6-v2


c:\Users\USER\Downloads\RAG_CRASH_COURSE\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2360.34it/s]


Model loaded successfully. Embedding Dimension: 384


C:\Users\USER\AppData\Local\Temp\ipykernel_26444\3017927795.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector store

In [8]:
class VectorStore:
    """
    Manages document embeddings in a ChromaDB vector store
    
    Chroma DB is an open-source vector database used to store & search embeddings
    """
    
    def __init__(self, collection_name: str="pdf_documents", persist_directory: str="../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        
        """
        self.collection_name = collection_name
        # whatever vector store is going to create will be saved in hard disk
        self.persist_directory = persist_directory
        self.client = None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create peristent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        
        except Exception as e:
            printf("Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} to vector store...")

        # Prepare data for ChromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document Content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding docs to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


### Text -> Embeddings -> VectorDB

In [10]:
texts=[doc.page_content for doc in chunks]

# Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

# Store in the vector db
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 58 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.48s/it]

Generated embeddings with shape: (58, 384)
Adding 58 to vector store...
Successfully added 58 documents to vector store
Total documents in collection: 58


### Retriever Pipeline 

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query:str, top_k: int=5, score_threshold: float=0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in the vector store
        try: 
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas=results['metadatas'][0]
                distances = results['distances'][0]
                ids=results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [14]:
rag_retriever.retrieve("What is attention is all you need?")

Retrieving documents for query: 'What is attention is all you need?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.85it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]